In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [3]:
import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
import numpy as np

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [5]:
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df


# ─────────────────────────────────────────────
# 2. BASIC CLEANING (applied to both train/test)
# ─────────────────────────────────────────────
def basic_cleaning(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Drop non-predictive ID
    df.drop(columns=["id"], inplace=True, errors="ignore")

    # Fix TotalCharges — blank strings for new customers
    df["TotalCharges_is_missing"] = df["TotalCharges"].isna().astype(int)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

    # Encode binary target
    if "Churn" in df.columns:
        df["Churn"] = df["Churn"].str.strip().str.lower().map({"yes": 1, "no": 0})

    return df


# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING (same for train/test)
# ─────────────────────────────────────────────
def feature_engineering(df):
    df = df.copy()

    # ── Interaction features ──
    df["Charges_x_Tenure"]       = df["MonthlyCharges"] * df["tenure"]
    df["ChargesPerMonth"]        = df["TotalCharges"] / (df["tenure"] + 1)
    df["ChargeIncreaseRate"]     = df["MonthlyCharges"] / (df["ChargesPerMonth"] + 1)

    # ── Risk score — high charge + short tenure + no contract ──
    df["IsMonthToMonth"]         = (df["Contract"] == "Month-to-month").astype(int)
    df["HighChargeShortTenure"]  = (
        (df["MonthlyCharges"] > df["MonthlyCharges"].median()) &
        (df["tenure"] < 12)
    ).astype(int)

    # ── Service adoption ──
    addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                  "TechSupport", "StreamingTV", "StreamingMovies"]
    df["NumAddons"]              = df[addon_cols].apply(lambda r: (r == "Yes").sum(), axis=1)
    df["HasNoAddons"]            = (df["NumAddons"] == 0).astype(int)
    df["AddonToChargeRatio"]     = df["NumAddons"] / (df["MonthlyCharges"] + 1)

    # ── Loyalty indicators ──
    df["TenureSquared"]          = df["tenure"] ** 2       # non-linear tenure effect
    df["IsNewCustomer"]          = (df["tenure"] <= 3).astype(int)
    df["IsLoyalCustomer"]        = (df["tenure"] >= 48).astype(int)

    # ── Payment risk ──
    df["IsAutoPay"] = df["PaymentMethod"].isin(
        ["Bank transfer (automatic)", "Credit card (automatic)"]
    ).astype(int)

    # ── Tenure bucket ──
    df["TenureBucket"] = pd.cut(
        df["tenure"], bins=[0, 12, 24, 48, 72],
        labels=["0-1yr", "1-2yr", "2-4yr", "4-6yr"],
        include_lowest=True
    ).astype(str)

    return df


# ─────────────────────────────────────────────
# 4. COLUMN DEFINITIONS
# ─────────────────────────────────────────────

# Already numeric — just scale
NUMERIC_COLS = ["tenure", "MonthlyCharges", "TotalCharges", "ChargesPerMonth", "NumAddons"]

# Yes/No binary columns
BINARY_COLS = ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"]

# Multi-class categoricals → One-Hot Encode
OHE_COLS = [
    "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
    "Contract", "PaymentMethod", "TenureBucket",
]


# ─────────────────────────────────────────────
# 5. CUSTOM BINARY ENCODER
# ─────────────────────────────────────────────
def encode_binary(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    X["gender"] = (X["gender"] == "Male").astype(int)
    yes_no_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"]
    for col in yes_no_cols:
        X[col] = (X[col] == "Yes").astype(int)
    return X


binary_transformer = FunctionTransformer(encode_binary)


# ─────────────────────────────────────────────
# 6. COLUMN TRANSFORMER
# ─────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_COLS),
        ("bin", binary_transformer, BINARY_COLS),
        (
            "ohe",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            OHE_COLS,
        ),
    ],
    remainder="drop",  # drop SeniorCitizen or handle separately below
)

# Note: SeniorCitizen is already 0/1 int — add to NUMERIC_COLS or BINARY_COLS as preferred
# Here we include it in numeric for scaling
NUMERIC_COLS_FULL = NUMERIC_COLS + ["SeniorCitizen"]

preprocessor_full = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_COLS_FULL),
        ("bin", binary_transformer, BINARY_COLS),
        (
            "ohe",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            OHE_COLS,
        ),
    ],
    remainder="drop",
)




In [6]:
# ─────────────────────────────────────────────
# TRAIN DATA
# ─────────────────────────────────────────────
df = load_data("/kaggle/input/competitions/playground-series-s6e3/train.csv")
df = basic_cleaning(df)
df = feature_engineering(df)

X = df.drop(columns=["Churn"])
y = df["Churn"]

# Split BEFORE preprocessing — keep a hold-out to evaluate AUC
from sklearn.model_selection import train_test_split

X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ─────────────────────────────────────────────
# TEST DATA — save id before cleaning drops it
# ─────────────────────────────────────────────
df_test  = load_data("/kaggle/input/competitions/playground-series-s6e3/test.csv")
test_ids = df_test["id"]                # ← save before basic_cleaning drops it
df_test  = basic_cleaning(df_test)
df_test  = feature_engineering(df_test)

# ─────────────────────────────────────────────
# PREPROCESSOR — fit on train only
# ─────────────────────────────────────────────
preprocessor_full.fit(X_train_raw)

X_train = preprocessor_full.transform(X_train_raw)
X_val   = preprocessor_full.transform(X_val_raw)     # validation — no fit
X_test  = preprocessor_full.transform(df_test)       # test — no fit




# Reset index after SMOTE (needed for Optuna CV indexing)
y_train = pd.Series(y_train).reset_index(drop=True)
X_train = np.array(X_train)

print("X_train shape:", X_train.shape)
print("X_val shape  :", X_val.shape)
print("X_test shape :", X_test.shape)

X_train shape: (475355, 46)
X_val shape  : (118839, 46)
X_test shape : (254655, 46)


In [7]:
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)
y_train.to_pickle("y_train.pkl")

In [8]:
X_train = np.load("/kaggle/working/X_train.npy")
y_train = pd.read_pickle("/kaggle/working/y_train.pkl")

In [9]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

np.save("X_tr.npy", X_tr)
np.save("X_val.npy", X_val)
y_tr.to_pickle("y_tr.pkl")
y_val.to_pickle(" y_val.pkl")

In [10]:
def objective_lr(trial):
    # --- 1. Search space (minimal but effective) ---
    C = trial.suggest_float("C", 1e-4, 100.0, log=True)

    model = LogisticRegression(
        C=C,
        penalty="l2",
        solver="lbfgs",      # fastest for l2
        max_iter=300,
        n_jobs=-1,
        random_state=42,
    )

    
    model.fit(X_tr, y_tr)
    preds = model.predict_proba(X_val)[:, 1]
    score = roc_auc_score(y_val, preds)

    # --- 3. Report score for pruning ---
    trial.report(score, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return score


In [11]:
print("\nTuning Logistic Regression...")
study_lr = optuna.create_study(direction="maximize",
                               sampler=optuna.samplers.TPESampler(seed=42),
                               pruner=optuna.pruners.MedianPruner())
study_lr.optimize(objective_lr, n_trials=80, show_progress_bar=True)

[I 2026-03-31 20:52:55,222] A new study created in memory with name: no-name-c3051300-9b5a-44ae-a745-dfada49c2afb



Tuning Logistic Regression...


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-03-31 20:52:59,789] Trial 0 finished with value: 0.9085409957424931 and parameters: {'C': 0.017670169402947963}. Best is trial 0 with value: 0.9085409957424931.
[I 2026-03-31 20:53:04,950] Trial 1 finished with value: 0.908603790650011 and parameters: {'C': 50.61576888752309}. Best is trial 1 with value: 0.908603790650011.
[I 2026-03-31 20:53:09,444] Trial 2 finished with value: 0.908626192040239 and parameters: {'C': 2.465832945854912}. Best is trial 2 with value: 0.908626192040239.
[I 2026-03-31 20:53:14,608] Trial 3 finished with value: 0.9086013888253242 and parameters: {'C': 0.39079671568228835}. Best is trial 2 with value: 0.908626192040239.
[I 2026-03-31 20:53:18,237] Trial 4 finished with value: 0.9076242963614272 and parameters: {'C': 0.0008632008168602544}. Best is trial 2 with value: 0.908626192040239.
[I 2026-03-31 20:53:21,991] Trial 5 pruned. 
[I 2026-03-31 20:53:24,691] Trial 6 pruned. 
[I 2026-03-31 20:53:29,210] Trial 7 finished with value: 0.9086055387573663 a

In [12]:
lr_params = study_lr.best_params.copy()
best_lr = LogisticRegression(**lr_params, solver="lbfgs", max_iter=1000, random_state=42)
best_lr.fit(X_train, y_train)

LogisticRegression(C=1.3715751458761123, max_iter=1000, random_state=42)

In [13]:
prob_lr_val  = best_lr.predict_proba(X_val)[:, 1]
np.save("lr_val_predict.npy",prob_lr_val)
auc_lr  = roc_auc_score(y_val, prob_lr_val)
print(f"Validation AUC — LR      : {auc_lr:.4f}")

Validation AUC — LR      : 0.9086


In [16]:
y_prob_final = best_lr.predict_proba(X_test)[:, 1]

np.save("lr_best_predict.npy", y_prob_final)

submission = pd.DataFrame({
    "id":    test_ids,
    "Churn": y_prob_final 
})

submission.to_csv("submission.csv", index=False)
print(submission["Churn"].value_counts())
print(submission.head(10))

Churn
0.354525    1
0.051789    1
0.000320    1
0.127411    1
0.003644    1
           ..
0.260778    1
0.800002    1
0.001322    1
0.475347    1
0.017059    1
Name: count, Length: 254655, dtype: int64
       id     Churn
0  594194  0.051789
1  594195  0.000320
2  594196  0.127411
3  594197  0.003644
4  594198  0.438557
5  594199  0.188861
6  594200  0.629316
7  594201  0.001384
8  594202  0.046368
9  594203  0.409091
